In [1]:
import logging
import torch
import torch.nn as nn
import torch.optim as optim
import torch_numopt
import numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader, TensorDataset
from sklearn.datasets import *
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error
from train_loop import train_loop

In [2]:
device = "cuda" if torch.cuda.is_available() else "cpu"

logging.basicConfig(level=logging.DEBUG)
# n_epoch = 1000
n_epoch = 10

In [3]:
class Net(nn.Module):
    def __init__(self, input_size, device="cpu"):
        super().__init__()
        self.f1 = nn.Linear(input_size, 10, device=device)
        self.f2 = nn.Linear(10, 20, device=device)
        self.f3 = nn.Linear(20, 20, device=device)
        self.f4 = nn.Linear(20, 10, device=device)
        self.f5 = nn.Linear(10, 1, device=device)

        self.activation = nn.ReLU()
        # self.activation = nn.Sigmoid()

    def forward(self, x):
        x = self.activation(self.f1(x))
        x = self.activation(self.f2(x))
        x = self.activation(self.f3(x))
        x = self.activation(self.f4(x))
        x = self.f5(x)

        return x

In [4]:
# X, y = load_diabetes(return_X_y=True, scaled=False)
# X, y = make_regression(n_samples=1000, n_features=100)
X, y = make_friedman1(n_samples=1000, noise=1e-2)

X_scaler = MinMaxScaler()
X = X_scaler.fit_transform(X)
X = torch.Tensor(X).to(device)

y_scaler = MinMaxScaler()
y = y_scaler.fit_transform(y.reshape((-1, 1)))
y = torch.Tensor(y).to(device)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)
print(X_train.shape, y_train.shape)
print(X_test.shape, y_test.shape)

torch_data = TensorDataset(X_train, y_train)
data_loader = DataLoader(torch_data, batch_size=10000)

torch.Size([800, 10]) torch.Size([800, 1])
torch.Size([200, 10]) torch.Size([200, 1])


In [5]:
model = Net(input_size=X.shape[1], device=device)
loss_fn = nn.MSELoss()

opt = torch_numopt.GradientDescentLS(model=model)

model, loss_history = train_loop(
    model,
    loss_fn,
    opt,
    data_loader,
    epochs=n_epoch,
    max_patience=50
)

INFO:torch_numopt.numerical_optimizer:No curvature info. used, returning gradient.
/home/eugenio/anaconda3/envs/torch_numopt/lib/python3.10/logging/__init__.py:368: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:836.)
  msg = msg % self.args
INFO:torch_numopt.line_search:Starting backtracking line search with initial guess of 1 with loss of 0.502451.
INFO:torch_numopt.line_search:Step was rejected.
DEBUG:torch_numopt.line_search:Iteration 0, new guess is 0.1 which yielded a loss of 0.121012.
INFO:torch_numopt.line_search:Step was accepted.
INFO:torch_numopt.line_search:Settled into lr = 0.1.
INFO:torch_numopt.numerical_optimizer:No curvature info. used, returning gradient.
INFO:torch_numopt.numerical_optimizer:Initial lr generated = 1 with method None and initial guess 1.
INFO:torch_numopt.line_search:St

epoch:  0, loss: 0.1987626999616623
epoch:  1, loss: 0.12101180851459503
epoch:  2, loss: 0.08037308603525162
epoch:  3, loss: 0.058704156428575516
epoch:  4, loss: 0.04722969979047775
epoch:  5, loss: 0.041236087679862976
epoch:  6, loss: 0.0381438322365284
epoch:  7, loss: 0.03656310960650444
epoch:  8, loss: 0.035758472979068756
epoch:  9, loss: 0.03534767031669617


In [6]:
model.eval()
pred_train = model.forward(X_train).cpu().detach()
pred_test = model.forward(X_test).cpu().detach()
print(f"Train metrics: R2 = {r2_score(pred_train, y_train.cpu())}")
print(f"Test metrics:  R2 = {r2_score(pred_test, y_test.cpu())}")

Train metrics: R2 = -779.2088455579902
Test metrics:  R2 = -737.1722994041671


In [11]:
model = Net(input_size=X.shape[1], device=device)
loss_fn = nn.MSELoss()

opt = torch_numopt.GradientDescentTR(model=model, radius_init=10)

model, loss_history = train_loop(
    model,
    loss_fn,
    opt,
    data_loader,
    epochs=1000,
    max_patience=50
)

INFO:root:Starting trust region loop with radius 10.
INFO:root:Finished trust region search, rho = -0.910964, final model radius = 2.5.
INFO:root:Starting trust region loop with radius 2.5.
INFO:root:Finished trust region search, rho = -0.910964, final model radius = 0.625.
INFO:root:Starting trust region loop with radius 0.625.
INFO:root:Finished trust region search, rho = 0.466147, final model radius = 0.625.
INFO:root:Starting trust region loop with radius 0.625.
INFO:root:Finished trust region search, rho = -0.928869, final model radius = 0.15625.
INFO:root:Starting trust region loop with radius 0.15625.
INFO:root:Finished trust region search, rho = 0.490146, final model radius = 0.15625.
INFO:root:Starting trust region loop with radius 0.15625.
INFO:root:Finished trust region search, rho = -0.941231, final model radius = 0.0390625.
INFO:root:Starting trust region loop with radius 0.0390625.
INFO:root:Finished trust region search, rho = 0.214796, final model radius = 0.00976562.
IN

epoch:  0, loss: 0.3707445561885834
epoch:  1, loss: 0.3707445561885834
epoch:  2, loss: 0.3707445561885834
epoch:  3, loss: 0.06399200856685638
epoch:  4, loss: 0.06399200856685638
epoch:  5, loss: 0.03655635565519333
epoch:  6, loss: 0.03655635565519333
epoch:  7, loss: 0.03615741804242134
epoch:  8, loss: 0.03581869974732399
epoch:  9, loss: 0.035738684237003326
epoch:  10, loss: 0.035697754472494125
epoch:  11, loss: 0.03563636913895607
epoch:  12, loss: 0.035564254969358444
epoch:  13, loss: 0.03548165038228035
epoch:  14, loss: 0.035402268171310425
epoch:  15, loss: 0.035336531698703766
epoch:  16, loss: 0.035277556627988815
epoch:  17, loss: 0.03522175922989845
epoch:  18, loss: 0.03516410291194916
epoch:  19, loss: 0.03510890156030655
epoch:  20, loss: 0.03505101054906845
epoch:  21, loss: 0.03499444201588631


INFO:root:Starting trust region loop with radius 0.00976562.
INFO:root:Finished trust region search, rho = 0.571624, final model radius = 0.00976562.
INFO:root:Starting trust region loop with radius 0.00976562.
INFO:root:Finished trust region search, rho = 0.5742, final model radius = 0.00976562.
INFO:root:Starting trust region loop with radius 0.00976562.
INFO:root:Finished trust region search, rho = 0.555939, final model radius = 0.00976562.
INFO:root:Starting trust region loop with radius 0.00976562.
INFO:root:Finished trust region search, rho = 0.551823, final model radius = 0.00976562.
INFO:root:Starting trust region loop with radius 0.00976562.
INFO:root:Finished trust region search, rho = 0.555143, final model radius = 0.00976562.
INFO:root:Starting trust region loop with radius 0.00976562.
INFO:root:Finished trust region search, rho = 0.574006, final model radius = 0.00976562.
INFO:root:Starting trust region loop with radius 0.00976562.
INFO:root:Finished trust region search, r

epoch:  22, loss: 0.034936558455228806
epoch:  23, loss: 0.03488041087985039
epoch:  24, loss: 0.03482263907790184
epoch:  25, loss: 0.03476741909980774
epoch:  26, loss: 0.03471166640520096
epoch:  27, loss: 0.0346553735435009
epoch:  28, loss: 0.03459693491458893
epoch:  29, loss: 0.03453740477561951
epoch:  30, loss: 0.03447573632001877
epoch:  31, loss: 0.03441271930932999
epoch:  32, loss: 0.03434805944561958
epoch:  33, loss: 0.03428160399198532
epoch:  34, loss: 0.03421318531036377
epoch:  35, loss: 0.03414255380630493
epoch:  36, loss: 0.03406914323568344
epoch:  37, loss: 0.0339936725795269
epoch:  38, loss: 0.03391486406326294
epoch:  39, loss: 0.033833324909210205
epoch:  40, loss: 0.03374858945608139


INFO:root:Finished trust region search, rho = 0.82175, final model radius = 0.00976562.
INFO:root:Starting trust region loop with radius 0.00976562.
INFO:root:Finished trust region search, rho = 0.84591, final model radius = 0.00976562.
INFO:root:Starting trust region loop with radius 0.00976562.
INFO:root:Finished trust region search, rho = 0.867926, final model radius = 0.00976562.
INFO:root:Starting trust region loop with radius 0.00976562.
INFO:root:Finished trust region search, rho = 0.896474, final model radius = 0.00976562.
INFO:root:Starting trust region loop with radius 0.00976562.
INFO:root:Finished trust region search, rho = 0.927597, final model radius = 0.00976562.
INFO:root:Starting trust region loop with radius 0.00976562.
INFO:root:Finished trust region search, rho = 0.948889, final model radius = 0.00976562.
INFO:root:Starting trust region loop with radius 0.00976562.
INFO:root:Finished trust region search, rho = 0.970392, final model radius = 0.00976562.
INFO:root:Sta

...

epoch:  951, loss: 0.008181053213775158
epoch:  952, loss: 0.008180957287549973
epoch:  953, loss: 0.008180862292647362
epoch:  954, loss: 0.008180767297744751
epoch:  955, loss: 0.00818067230284214
epoch:  956, loss: 0.008180578239262104
epoch:  957, loss: 0.008180483244359493
epoch:  958, loss: 0.008180387318134308
epoch:  959, loss: 0.008180291391909122
epoch:  960, loss: 0.008180196397006512
epoch:  961, loss: 0.008180102333426476
epoch:  962, loss: 0.00818000640720129
epoch:  963, loss: 0.008179915137588978
epoch:  964, loss: 0.008179832249879837
epoch:  965, loss: 0.008179750293493271
epoch:  966, loss: 0.008179668337106705
epoch:  967, loss: 0.008179585449397564
epoch:  968, loss: 0.008179504424333572
epoch:  969, loss: 0.008179422467947006


INFO:root:Finished trust region search, rho = 1.01163, final model radius = 0.000610352.
INFO:root:Starting trust region loop with radius 0.000610352.
INFO:root:Finished trust region search, rho = 1.02326, final model radius = 0.000610352.
INFO:root:Starting trust region loop with radius 0.000610352.
INFO:root:Finished trust region search, rho = 1.01163, final model radius = 0.000610352.
INFO:root:Starting trust region loop with radius 0.000610352.
INFO:root:Finished trust region search, rho = 1, final model radius = 0.000610352.
INFO:root:Starting trust region loop with radius 0.000610352.
INFO:root:Finished trust region search, rho = 1.02353, final model radius = 0.000610352.
INFO:root:Starting trust region loop with radius 0.000610352.
INFO:root:Finished trust region search, rho = 1.01176, final model radius = 0.000610352.
INFO:root:Starting trust region loop with radius 0.000610352.
INFO:root:Finished trust region search, rho = 1.01176, final model radius = 0.000610352.
INFO:root:S

epoch:  970, loss: 0.00817934237420559
epoch:  971, loss: 0.008179261349141598
epoch:  972, loss: 0.008179179392755032
epoch:  973, loss: 0.00817909836769104
epoch:  974, loss: 0.008179018273949623
epoch:  975, loss: 0.008178937248885632
epoch:  976, loss: 0.008178857155144215
epoch:  977, loss: 0.008178777061402798
epoch:  978, loss: 0.008178697898983955
epoch:  979, loss: 0.008178617805242538
epoch:  980, loss: 0.008178538642823696
epoch:  981, loss: 0.00817845854908228
epoch:  982, loss: 0.008178380317986012
epoch:  983, loss: 0.00817830115556717
epoch:  984, loss: 0.008178221993148327
epoch:  985, loss: 0.008178142830729485
epoch:  986, loss: 0.008178063668310642
epoch:  987, loss: 0.0081779845058918
epoch:  988, loss: 0.008177905343472958
epoch:  989, loss: 0.008177826181054115
epoch:  990, loss: 0.008177747949957848


INFO:root:Starting trust region loop with radius 0.000610352.
INFO:root:Finished trust region search, rho = 1.01205, final model radius = 0.000610352.
INFO:root:Starting trust region loop with radius 0.000610352.
INFO:root:Finished trust region search, rho = 1, final model radius = 0.000610352.
INFO:root:Starting trust region loop with radius 0.000610352.
INFO:root:Finished trust region search, rho = 1, final model radius = 0.000610352.
INFO:root:Starting trust region loop with radius 0.000610352.
INFO:root:Finished trust region search, rho = 1.0122, final model radius = 0.000610352.
INFO:root:Starting trust region loop with radius 0.000610352.
INFO:root:Finished trust region search, rho = 1.02439, final model radius = 0.000610352.
INFO:root:Starting trust region loop with radius 0.000610352.
INFO:root:Finished trust region search, rho = 1.01235, final model radius = 0.000610352.
INFO:root:Starting trust region loop with radius 0.000610352.
INFO:root:Finished trust region search, rho =

epoch:  991, loss: 0.008177668787539005
epoch:  992, loss: 0.008177590556442738
epoch:  993, loss: 0.008177513256669044
epoch:  994, loss: 0.008177436888217926
epoch:  995, loss: 0.008177359588444233
epoch:  996, loss: 0.008177281357347965
epoch:  997, loss: 0.008177204988896847
epoch:  998, loss: 0.008177128620445728
epoch:  999, loss: 0.00817705038934946


In [12]:
model.eval()
pred_train = model.forward(X_train).cpu().detach()
pred_test = model.forward(X_test).cpu().detach()
print(f"Train metrics: R2 = {r2_score(pred_train, y_train.cpu())}")
print(f"Test metrics:  R2 = {r2_score(pred_test, y_test.cpu())}")

Train metrics: R2 = 0.700660467979028
Test metrics:  R2 = 0.6156296899641309
